# CRED Collections — Analysis Notebook
**Question:** did recovery really grow 11%? **Answer:** no — flat once calendar-normalised and formally tested.

**How to run anywhere (no hardcoded user paths):**
```bash
pip install -r requirements.txt
python solution/build_golden.py            # raw -> golden (relative paths; or CRED_DATA_DIR=...)
python solution/run_analysis.py            # tests + stratified + executed DiD
jupyter lab solution/analysis_notebook.ipynb
```
This notebook narrates the reasoning and displays tracked outputs. Heavy lifting lives in `build_golden.py` / `run_analysis.py`; cells below verify, not duplicate.

In [ ]:
from pathlib import Path
import json, os
HERE = Path.cwd()
# Auto-detect: notebook lives in solution/; data is a sibling (either name)
CANDS = [HERE, HERE/"solution", HERE.parent, Path(os.environ.get("CRED_DATA_DIR",""))]
DATA = next((c for c in [
    Path(os.environ["CRED_DATA_DIR"]) if "CRED_DATA_DIR" in os.environ else None,
    HERE/"collections_30k_dataset (4)", HERE/"collections_30k_dataset",
    HERE/"solution"/".."/"collections_30k_dataset (4)",
    HERE.parent/"collections_30k_dataset (4)", HERE.parent/"collections_30k_dataset",
] if c and Path(c).exists()), None)
GOLD = next((c for c in [
    Path(os.environ["CRED_OUT_DIR"]) if "CRED_OUT_DIR" in os.environ else None,
    HERE/"golden", HERE/"solution"/"golden", HERE.parent/"solution"/"golden",
] if c and Path(c).exists()), None)
print("DATA :", DATA)
print("GOLD :", GOLD)
assert DATA and GOLD, "Put raw CSVs next to solution/ or set CRED_DATA_DIR"
import pandas as pd
stats = json.loads((GOLD/"stats.json").read_text())
print(f"golden SUCCESS {stats['gold_success_n']} (raw {stats['raw_success_n']}, removed {stats['removed_n']})")

## Q1 — What happened? (raw vs golden vs per-day)
Logic: money = SUCCESS-only, `payment_id`-deduped, Aug excluded (partial). Calendar normalisation (`per_day = total / days_in_month`) kills the Feb(28d)→Mar(31d) illusion before any modelling.

In [ ]:
import pandas as pd
kpi = pd.read_csv(GOLD/"monthly_kpi.csv", index_col="month")
print(kpi.round(0).to_string())
print("\nMar raw MoM: +%.2f%% -> per-day: %+.2f%%" % (
    kpi.loc["2026-03","mom_raw"], kpi.loc["2026-03","mom_perday"]))
print("Verdict: volume (+10.2% rows) + calendar explain Mar; efficiency (+0.8%/row) flat.")

## Q2 — Forensics A–H (counts, not assertions)
A dups · B TXN-reuse attribution risk · C 3 timezones→IST · D no vendor-code shift · E 1,000 agents × ~30 snapshots · F easy-mix (fresh DPD 0–30 Mar spike) · G 6,656 never-targeted (coverage gap, NOT proven fruit) · **H priority 1–10 flat (NEW)**.

In [ ]:
import pandas as pd
from scipy import stats as st
pay_raw = pd.read_csv(DATA/"payments.csv", low_memory=False, usecols=["payment_id","payment_reference","payment_status"])
print(f"A: full-dup rows report §4; ID dups={pay_raw.duplicated('payment_id').sum()}, "
      f"TXN reuse={pay_raw[pay_raw.payment_reference.notna()].duplicated('payment_reference').sum()}")
print("C:", pd.read_csv(DATA/"calls.csv", usecols=["timezone"])["timezone"].value_counts().to_dict())
print("D:", pd.read_csv(DATA/"call_dispositions.csv", usecols=["disposition_version"])["disposition_version"].value_counts().to_dict())
print("E: agent_ids =", pd.read_csv(DATA/"agents.csv", usecols=["agent_id"])["agent_id"].nunique(), "/ 30000 rows")
print("G: never-targeted =",
      30000 - pd.read_csv(DATA/"daily_targeting.csv", usecols=["account_id"])["account_id"].nunique())
h = __import__("json").loads((GOLD/"forensic_h.json").read_text())
print(f"H: overall 7d={h['overall_7d']:.4f}, chi2={h['chi2']:.2f} p={h['p']:.3f} -> {h['verdict']}")

## Q3 — Formal tests (why “FACT” is defensible)
Point estimates (+0.29%) don't survive cross-examination; tests do. Welch (unequal-variance means), permutation (exact null by shuffling days), bootstrap (CI for the lift). Same for targeted-vs-never (two-proportion z + DPD×risk MH) and priority (χ²).

In [ ]:
import json
t = json.loads((GOLD/"feb_mar_test.json").read_text())
print(f"Feb {t['feb_mean']:,.0f}/d (n={t['feb_n']}) vs Mar {t['mar_mean']:,.0f}/d (n={t['mar_n']})")
print(f"Welch t={t['welch_t']:.3f} p={t['welch_p']:.3f} | perm p={t['perm_p']:.3f} | boot CI {t['boot_ci_pct'][0]:+.2f}%..{t['boot_ci_pct'][1]:+.2f}% -> {t['verdict']}")
s = json.loads((GOLD/"stratified_uplift.json").read_text())
print(f"Targeted {s['targeted_rate']:.4f} vs never {s['never_rate']:.4f} diff {s['diff']:+.4f} p={s['p']:.3f} CI [{s['ci'][0]:+.4f},{s['ci'][1]:+.4f}]; MH {s['mh_diff']:+.4f}")

## Q4 — Counterfactual, EXECUTED (PSM + DiD)
Design: Mar cutover; treat = Mar targeting under v2/v3 (new logic), control = Mar targeting legacy/v1-only, 1:1 PSM (caliper 0.05) on DPD/outstanding/risk/loan; pre = Jan–Feb ever-paid, post = Apr–May ever-paid (Mar skipped). Expect null/wide — the weak first stage *is* the finding; it forces the prospective 50/50 holdout.

In [ ]:
import json
d = json.loads((GOLD/"counterfactual_did.json").read_text())
r = d["rates"]
print(d["design"])
print(f"pairs={d['n_pairs']} (treat pool {d['n_treat_pool']}, control pool {d['n_control_pool']})")
print(f"treat {r['treat_pre']:.4f}->{r['treat_post']:.4f} | control {r['control_pre']:.4f}->{r['control_post']:.4f}")
print(f"DiD {d['did_pp']:+.4f} pp (rel {d['did_rel']:+.1%}), 95% CI [{d['boot_ci_pp'][0]:+.4f},{d['boot_ci_pp'][1]:+.4f}]")
print("parallel Jan->Feb:", {k: round(v,4) for k,v in d["parallel_pre_trends"].items()})
print("balance SMD:", {k: round(v["smd"],3) for k,v in d["balance_smd"].items()})
print("->", d["verdict"])

## Where to invest ₹10 Cr? (assumption-gated scenarios)
Empirical lift for *current* targeting ≈ 0 (MH −0.65pp, p=0.35); priority has no signal (p=0.345). So any *new-score* lift is a hypothesis with a capped experiment, not a measurement. Base ₹126.8Cr golden Jan–Jul.

In [ ]:
base = 126.8
for name, lift in [("pessimistic (holdout fails)", 0.00), ("base (new-score assumption)", 0.04), ("upside (holdout-gated)", 0.10)]:
    inc = base * lift
    print(f"{name:32s} lift {lift:5.1%} -> +₹{inc:5.1f} Cr/yr  net ROI x{(inc-1.2)/1.2:+.1f} on assumed ₹1.2Cr")
print("50% volume = holdout split (design choice). Kill-switch: stop if <3% at 4-wk interim.")

## Repro map
- `build_golden.py` — sole generator of `golden/*.csv + stats.json + daily_recovery.csv` (run first).
- `run_analysis.py` — regenerates `feb_mar_test / stratified_uplift / forensic_h / counterfactual_did` JSONs.
- `sql/01–05` — warehouse mirror of the same logic (PKs, IST, 7-day PTP, DiD scaffold + executed spec).
- `DATA_QUALITY_REPORT.md` — standalone DQ deliverable. `EXECUTIVE_MEMO_1PAGE.md` — 60-second CEO read; `memo.md` + appendix hold the stats.